# Masterclass 3: Sectoral Decision Intelligence Across 10 Strategic Sectors
### *Operational Multi-Criteria Spatial Prioritization Across Healthcare, Commerce, Education, WASH Utilities, Public Safety, and Governance*

---

## 1. Overview: The 10 Strategic Sectors in Nigerian Ward Spatial Intelligence

Spatial statistics transforms raw Earth Observation data and facility registries into operational strategy. We operationalize this across **10 active strategic sectors**:

| # | Strategic Sector | Empirical Data Variables | Real-World Policy & Enterprise Decision | Spatial Method Applied |
| :-: | :--- | :--- | :--- | :--- |
| **1** | **Public Health** | `health_facilities_count`, `pop_2025_sum` | Eliminate Healthcare Deserts (>17k pop, 0 clinics) | Point-in-Polygon Joins & Buffer Siting |
| **2** | **Wealth & Poverty** | `rwi_mean`, `rwi_std` | Direct capital to structural poverty corridors | Meta RWI Zonal Stats & LISA Coldspots |
| **3** | **Commerce & Retail** | `markets_count`, `rwi_mean` | Siting supermarkets & distribution warehouses | 4-Quadrant Catchment Matrix |
| **4** | **Demographics** | `pop_2025_sum`, `population_density_per_sqkm` | Capacity planning & congestion relief | Gridded Demographic Aggregation |
| **5** | **Cultural Cohesion** | `churches_count`, `mosques_count` | Peacebuilding & inter-faith civic campaigns | Shannon Entropy Diversity Index |
| **6** | **WASH Utilities** | `water_points_count`, `water_rate` | Eradicate clean water inequality & boreholes | Lorenz Curves & Gini Coefficients |
| **7** | **Education** | `schools_count`, `pop_2025_sum` | School catchment deficits & classroom siting | School density per 10k school-age pop |
| **8** | **Public Safety** | `police_stations_count`, `fire_stations_count` | Emergency station response sheds & safety | Service Area Sheds & Outlier Analysis |
| **9** | **Epidemiology** | `malaria_prevalence_pct`, `malaria_annual_cases` | High-transmission vector control corridors | Spatial Lag Regression & LISA Hotspots |
| **10** | **Governance & Elections** | `ward_priority_index`, INEC Polling Units | Polling unit access mitigation & MCDA budgeting | Multi-Criteria Decision Analysis (WPI) |


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import libpysal
import esda

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')


In [ ]:
DATA_PATH = '../data/processed/nigeria_wards_master.parquet'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/processed/nigeria_wards_master.parquet'

gdf = gpd.read_parquet(DATA_PATH)
gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty].copy()
gdf.reset_index(drop=True, inplace=True)

gdf['rwi_mean'] = gdf['rwi_mean'].fillna(gdf['rwi_mean'].median())
gdf['pop_2025_sum'] = gdf['pop_2025_sum'].fillna(gdf['pop_2025_sum'].median())
gdf['health_rate'] = (gdf['health_facilities_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['market_rate'] = (gdf['markets_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['water_rate'] = (gdf['water_points_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['school_rate'] = (gdf['schools_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['police_rate'] = (gdf['police_stations_count'] / (gdf['pop_2025_sum'] + 100)) * 10000

print(f"Loaded {len(gdf):,} wards across {gdf['statename'].nunique()} states.")


\
## 2. Measuring Spatial Inequality Across Sectors: Lorenz Curves & Gini Coefficients

To quantify how unequally infrastructure is distributed across wards, we compute the **Gini Coefficient ($G$)**:
$$
G = rac{\sum_{i=1}^n \sum_{j=1}^n |x_i - x_j|}{2 n^2 ar{x}}
$$
- $G = 0.0$: Perfect spatial equality (every ward has an identical share).
- $G = 1.0$: Total spatial concentration (one ward has everything).


In [ ]:
def gini_coefficient(values):
    vals = np.sort(np.asarray(values, dtype=np.float64))
    n = len(vals)
    if n == 0 or np.all(vals == 0):
        return 0.0
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * vals) - (n + 1) * np.sum(vals)) / (n * np.sum(vals)))

def lorenz_curve(values):
    vals = np.sort(np.asarray(values, dtype=np.float64))
    cum_vals = np.cumsum(vals) / np.sum(vals)
    cum_pop = np.linspace(0, 1, len(vals))
    return cum_pop, cum_vals

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Line of Perfect Equality (G = 0.0)')

sector_facilities = [
    ('health_facilities_count', 'Primary Health Clinics', '#e63946'),
    ('markets_count', 'Commercial Markets', '#2a9d8f'),
    ('water_points_count', 'Clean Water Points (WASH)', '#457b9d'),
    ('schools_count', 'Primary & Secondary Schools', '#f4a261')
]

for col, lbl, color in sector_facilities:
    g = gini_coefficient(gdf[col].values)
    px, py = lorenz_curve(gdf[col].values)
    ax.plot(px, py, label=f"{lbl} (Gini = {g:.3f})", color=color, lw=2.2)

ax.set_title("Spatial Inequality Across Infrastructure Sectors (Lorenz Curves)", fontsize=13, fontweight='bold')
ax.set_xlabel("Cumulative Proportion of Administrative Wards")
ax.set_ylabel("Cumulative Proportion of Facilities")
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()


\
## 3. Cross-Sector Coupling: Education & Emergency Public Safety Gaps

We examine infrastructure coverage across Education (Schools per 10k) and Public Safety (Police & Fire stations).
Wards with large populations but zero schools or emergency services represent critical civic vulnerability zones.


In [ ]:
zero_schools = (gdf['schools_count'] == 0) & (gdf['pop_2025_sum'] > gdf['pop_2025_sum'].median())
zero_police = (gdf['police_stations_count'] == 0) & (gdf['pop_2025_sum'] > gdf['pop_2025_sum'].median())

print("=== CIVIC & EMERGENCY SERVICE SHORTAGE WARDS ===")
print(f"Wards with > Median Pop and 0 Registered Schools: {zero_schools.sum():,} wards ({zero_schools.sum()/len(gdf)*100:.1f}%)")
print(f"Wards with > Median Pop and 0 Registered Police:  {zero_police.sum():,} wards ({zero_police.sum()/len(gdf)*100:.1f}%)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

gdf.plot(column='school_rate', cmap='Blues', legend=True, ax=ax1,
         legend_kwds={'label': 'Schools per 10,000 Residents', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
ax1.set_title("A. Education Infrastructure: Schools Access Rate", fontsize=12, fontweight='bold')
ax1.axis('off')

gdf.plot(column='police_rate', cmap='Purples', legend=True, ax=ax2,
         legend_kwds={'label': 'Police Stations per 10,000 Residents', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
ax2.set_title("B. Public Safety Infrastructure: Police Coverage Rate", fontsize=12, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()


\
## 4. Multi-Criteria Decision Analysis (MCDA): The Ward Priority Index (WPI)

To convert multi-sector datasets into a single actionable capital allocation tool, we use MCDA:
1. **Min-Max Feature Scaling:**
$$
	ilde{x}_i = rac{x_i - \min(x)}{\max(x) - \min(x)}
$$
2. **Deficit Inversion:** Shortages are inverted so that larger scores represent higher deprivation.
3. **Composite Scoring:**
$$
	ext{WPI}_i = 0.30 \cdot 	ext{PovertyDeficit}_i + 0.30 \cdot 	ext{HealthDeficit}_i + 0.20 \cdot 	ext{WaterDeficit}_i + 0.20 \cdot 	ext{PopWeight}_i
$$


In [ ]:
def min_max(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-8)

poverty_def = min_max(-gdf['rwi_mean'])
health_def = min_max(1 / (gdf['health_rate'] + 0.1))
water_def = min_max(1 / (gdf['water_rate'] + 0.1))
pop_wt = min_max(np.log1p(gdf['pop_2025_sum']))

gdf['ward_priority_index'] = 0.30 * poverty_def + 0.30 * health_def + 0.20 * water_def + 0.20 * pop_wt

gdf['action_tier'] = pd.qcut(
    gdf['ward_priority_index'],
    q=4,
    labels=[
        'Tier 4: Mature / Self-Sustaining',
        'Tier 3: Moderate Support Needed',
        'Tier 2: High Investment Priority',
        'Tier 1: Critical Emergency Intervention'
    ]
)

tier_colors = {
    'Tier 1: Critical Emergency Intervention': '#d90429',
    'Tier 2: High Investment Priority': '#f77f00',
    'Tier 3: Moderate Support Needed': '#fcbf49',
    'Tier 4: Mature / Self-Sustaining': '#2a9d8f'
}

fig, ax = plt.subplots(figsize=(12, 8))
for t, col in tier_colors.items():
    sub = gdf[gdf['action_tier'] == t]
    sub.plot(color=col, ax=ax, label=f"{t} (n={len(sub):,})", linewidth=0.1, edgecolor='white')

ax.set_title("National Ward Priority Action Tiers (MCDA Allocation Map)", fontsize=13, fontweight='bold')
ax.axis('off')
tier_patches = [mpatches.Patch(color=c, label=f"{l} (n={(gdf['action_tier'] == l).sum():,})") for l, c in tier_colors.items()]
ax.legend(handles=tier_patches, loc='lower left', frameon=True, facecolor='white', framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
state_avg_wpi = gdf.groupby('statename')['ward_priority_index'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=state_avg_wpi.values, y=state_avg_wpi.index, palette='Reds_r', ax=ax)
ax.set_title("State-by-State Average Ward Priority Index (WPI Vulnerability)", fontsize=13, fontweight='bold')
ax.set_xlabel("Average Ward Priority Index (Higher = Greater Need)")
plt.tight_layout()
plt.show()


In [ ]:
top_15 = gdf.sort_values(by='ward_priority_index', ascending=False).head(15).copy()
matrix_df = top_15[['statename', 'wardname', 'rwi_mean', 'health_rate', 'water_rate', 'school_rate', 'pop_2025_sum']].set_index(['statename', 'wardname'])
norm_matrix = (matrix_df - matrix_df.mean()) / matrix_df.std()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(norm_matrix, annot=True, fmt=".2f", cmap='coolwarm_r', center=0, ax=ax,
            cbar_kws={'label': 'Normalized z-score (Red = Acute Deficit)'})
ax.set_title("Multi-Sector Deficit Profile: Top 15 Priority Wards in Nigeria", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


\
## 5. Institutional Governance & Capital Allocation Playbook

1. **Precision Budgeting:** Fund allocation must shift from flat LGA-level grants to ward-level priority tiers.
2. **Multi-Sector Bundling:** Interventions in Tier 1 wards must co-locate water boreholes, primary clinics, and micro-retail support.
3. **Electoral Logistics Alignment:** Utilizing the WPI priority tiers to optimize INEC polling unit access sheds, reducing voter travel burden in isolated rural communities.


## Primary Data Sources & Key References

### Primary Geospatial Data Sources
- **Administrative Ward Boundaries:** GRID3 Nigeria Admin-3 Wards (9,308 polygons): [https://grid3.gov.ng/datasets/nigeria/administrative-boundaries](https://grid3.gov.ng/datasets/nigeria/administrative-boundaries)
- **Relative Wealth Index (RWI):** Meta AI Research & UC Berkeley micro-wealth estimates: [https://data.humdata.org/dataset/relative-wealth-index](https://data.humdata.org/dataset/relative-wealth-index)
- **Demographic Population Counts:** WorldPop 2025 Gridded Population Projections: [https://hub.worldpop.org/geodata/listing?id=29](https://hub.worldpop.org/geodata/listing?id=29)
- **Points of Interest Registries:** GRID3 Nigeria Health Clinics, Markets, Water Points, Police, Religious Centers: [https://grid3.gov.ng/datasets](https://grid3.gov.ng/datasets)
- **Disease Epidemiology:** Malaria Atlas Project (MAP) Plasmodium falciparum $Pf\text{PR}_{2-10}$: [https://malariaatlas.org/](https://malariaatlas.org/)
- **Electoral Infrastructure:** INEC Polling Units Location Registry: [https://irev.inecnigeria.org](https://irev.inecnigeria.org)

### Methodological References & Literature
1. **Anselin, L. (1988).** *Spatial Econometrics: Methods and Models*. Kluwer Academic Publishers.
2. **Anselin, L. (1995).** Local Indicators of Spatial Association -- LISA. *Geographical Analysis*, 27(2), 93-115.
3. **Chi, G., Fang, H., Chatterjee, S., & Blumenstock, J. E. (2022).** Micro-estimate of wealth for all low- and middle-income countries. *PNAS*, 119(3), e2113658119.
4. **Rey, S. J., & Anselin, L. (2007).** PySAL: A Python library for spatial analytical methods. *The Review of Regional Studies*, 37(1), 5-27.
5. **Tobler, W. R. (1970).** A computer movie simulating urban growth in the Detroit region. *Economic Geography*, 46(sup1), 234-240.
6. **Weiss, D. J., et al. (2019).** Mapping the global prevalence, incidence, and mortality of Plasmodium falciparum, 2000-17. *The Lancet*, 394(10195), 322-331.
